# Mindicator MCP + smolagents Chatbot Demo

A beginner-friendly walkthrough: connect an AI agent to our **Mindicator** transit database through **MCP**, then chat with it.

**What you will build**

```text
You (chat) → smolagents (LLM brain) → MCP tools → Mindicator SQLite
```

**Prerequisites**
1. Terminal A: start the MCP server → `uv run mindicator-mcp`
2. An API key for an LLM (OpenAI shown below; easy to swap)
3. Install demo deps once: `uv sync --extra demo`

## 1. Big picture (60 seconds)

| Piece | Role |
|-------|------|
| **SQLite DB** | Mumbai local trains, buses, fares (`mumbai_mindicator.sqlite`) |
| **MCP server** | Exposes 3 tools over HTTP: `health_check`, `get_schema`, `execute_sql` |
| **smolagents** | Tiny agent framework — LLM decides which tools to call |
| **This notebook** | Client that talks to the MCP server and runs a chatbot |

MCP = **Model Context Protocol**. Think of it as a USB plug for AI tools: any MCP client can use our server without custom glue code.

## 2. Install / imports

Run this cell once after `uv sync --extra demo`.

In [1]:
import os
from getpass import getpass

from dotenv import load_dotenv
from smolagents import MCPClient, OpenAIModel, ToolCallingAgent

load_dotenv()  # loads .env from the project root if present
print("imports ok")

c:\Programming\Repo\mindicatormcp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports ok


## 3. Check the MCP server is up

In a **separate terminal** (keep it running):

```bash
uv run mindicator-mcp
```

You should see something like: `http://127.0.0.1:8000/mcp`.

Our notebook connects to that URL.

In [2]:
MCP_URL = os.getenv("MCP_URL", "http://127.0.0.1:8000/mcp")

# HTTP config for smolagents → our FastMCP server
mcp_config = {
    "url": MCP_URL,
    "transport": "streamable-http",  # matches FastMCP transport="http"
}
print("Will connect to:", MCP_URL)

Will connect to: http://127.0.0.1:8000/mcp


## 4. Choose an LLM

smolagents needs a model. Below we use **OpenAI** (simple for demos).

Set `OPENAI_API_KEY` in your `.env`, or paste it when prompted.

> Tip: `gpt-4o-mini` is cheap and good enough for this demo.

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

model = OpenAIModel(model_id="gpt-4o-mini")
print("model ready:", model.model_id)

## 5. Connect and list MCP tools

`MCPClient` opens a connection to our server and turns MCP tools into smolagents tools.

You should see: `health_check`, `get_schema`, `execute_sql`.

In [ ]:
mcp_client = MCPClient(mcp_config, structured_output=True)
tools = mcp_client.get_tools()

for t in tools:
    print(f"- {t.name}: {t.description[:80]}...")

print(f"\n{len(tools)} tools loaded")

## 6. Build a small agent

`ToolCallingAgent` = beginner-friendly. The LLM **calls tools** (no Python code generation).

Flow for one question:
1. User asks in English
2. Agent may call `execute_sql` (and maybe `get_schema`)
3. Agent answers using the tool results

In [ ]:
agent = ToolCallingAgent(
    tools=tools,
    model=model,
    max_steps=8,  # stop after a few tool rounds
)
print("agent ready")

## 7. First question (one-shot)

Ask something the DB can answer. Watch the agent pick tools.

In [ ]:
question = "How do I get from Churchgate to Thane on the local train?"
answer = agent.run(question)
print("\n=== ANSWER ===")
print(answer)

## 8. Try more sample questions

Uncomment one and run.

In [ ]:
samples = [
    "What is the auto rickshaw fare for about 5 km at night?",
    "List a few stops on BEST bus route 1(Up).",
    "Find stations whose name contains BANDRA.",
    "Is the Mindicator database healthy, and which city/version is it?",
]

# Change the index 0..3 to try another question
q = samples[0]
print("Q:", q)
print("\n=== ANSWER ===")
print(agent.run(q))

## 9. Simple chatbot loop

Type questions interactively. Type `quit` to stop.

> In Jupyter, `input()` works in the notebook. For a polished UI later, you could wrap the same agent in Gradio/Streamlit — same MCP tools.

In [ ]:
print("Mindicator chatbot (type 'quit' to exit)\n")

while True:
    user = input("You: ").strip()
    if not user:
        continue
    if user.lower() in {"quit", "exit", "q"}:
        print("Bye!")
        break

    reply = agent.run(user)
    print("Bot:", reply)
    print()

## 10. Cleanup

Always close the MCP client when you are done.

In [ ]:
mcp_client.disconnect()
print("MCP client closed")

## Recap

1. **MCP server** hosts tools over HTTP (`uv run mindicator-mcp`)
2. **smolagents `MCPClient`** loads those tools
3. **`ToolCallingAgent`** + an LLM = chatbot that can query Mumbai transit data
4. The agent writes **SQL** via `execute_sql` (read-only + LIMIT for safety)

### Optional: one-shot style with a context manager

```python
with MCPClient(mcp_config, structured_output=True) as tools:
    agent = ToolCallingAgent(tools=tools, model=model)
    print(agent.run("Find ferry services from Gorai"))
```

### Troubleshooting

| Problem | Fix |
|---------|-----|
| Connection refused | Start `uv run mindicator-mcp` first |
| Auth / API errors | Check `OPENAI_API_KEY` |
| Empty / wrong answers | Ask the agent to use `get_schema`, or try a clearer question |
| Import errors | `uv sync --extra demo` then select this venv as the notebook kernel |